In [5]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from evisionary_common import MISSING_LIKE, SOURCE_PRIORITY

# ===============================================================
# 0. CONFIG
# ===============================================================

print("=" * 88)
print(" Initiating EVisionary Database Harmonization")
print("=" * 88)

BASE_DIR = Path("/Users/sogand")
OUTPUT_DIR = Path("./EVisionary_outputs_keyB")
AUDIT_DIR = OUTPUT_DIR / "Validation_Audits"
OUTPUT_PARQUET = OUTPUT_DIR / "unified_EVmetadata_keyB.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment-level metadata files (joined onto cargo via EXPERIMENT_ID)
VESICLEPEDIA_EXPERIMENT = "VESICLEPEDIA_EXPERIMENT_DETAILS_5.1.txt"
EXOCARTA_EXPERIMENT = "ExoCarta_experiment_details_6.txt"
EVTRACK_FILE = "evtrack_aug2025.xlsx"

# ---------------------------------------------------------------
# Multi-cargo definition.
# Each cargo file declares which native column holds the working
# identifier. gene_details files are intentionally EXCLUDED because
# they are reference lookup tables (no EXPERIMENT_ID, no cargo
# observation), not per-experiment cargo records.
# ---------------------------------------------------------------
CARGO_FILES = {
    "Vesiclepedia": [
        {"file": "Vesiclepedia_protein_mRNA_details.txt", "id_col": "GENE_SYMBOL"},
        {"file": "VESICLEPEDIA_MIRNA_DETAILS_5.1.txt",     "id_col": "MIRNA_ID"},
        {"file": "VESICLEPEDIA_LIPID_DETAILS_5.1.txt",     "id_col": "LIPID_ID"},
    ],
    "ExoCarta": [
        {"file": "ExoCarta_protein_mRNA_details_6.txt", "id_col": "GENE_SYMBOL"},
        {"file": "ExoCarta_miRNA_details_6.txt",        "id_col": "MIRNA_ID"},
        {"file": "ExoCarta_lipid_details_6.txt",        "id_col": "LIPID_ID"},
    ],
}

# CONTENT_TYPE placeholder values that do not represent valid cargo.
INVALID_CONTENT_TYPES = {"notfound", "not found", "unmapped", "unknown", ""}

# ===============================================================
# PRIMARY DEDUP POLICY = key_B_plus_species
# ===============================================================
PRIMARY_DEDUP_KEY = [
    "pmid", "sample_name", "working_id", "molecule_type_norm", "method", "species", "source"
]

DEDUP_KEY_SENSITIVITY = {
    "key_A_without_species": ["pmid", "sample_name", "working_id", "molecule_type_norm", "method", "source"],
    "key_B_primary_plus_species": ["pmid", "sample_name", "working_id", "molecule_type_norm", "method", "species", "source"],
    "key_C_plus_species_disease_vesicle": [
        "pmid", "sample_name", "working_id", "molecule_type_norm",
        "method", "species", "disease", "vesicle", "source"
    ]
}

MAPPING_SPEC = {
    "Vesiclepedia": {
        "source_type": "cargo+experiment",
        "record_granularity": "cargo_level",
        "experiment_file": VESICLEPEDIA_EXPERIMENT,
        "join_key": "EXPERIMENT_ID",
        "required_experiment_headers": [
            "EXPERIMENT_ID", "PUBMED_ID", "SPECIES", "SAMPLE_SOURCE",
            "YEAR", "ISOLATION_METHOD", "VESICLE_TYPE"
        ],
        "canonical_map": {
            "pmid": "PUBMED_ID",
            "sample_name": "SAMPLE_SOURCE",
            "working_id": "WORKING_ID_RAW",
            "molecule_type_raw": "CONTENT_TYPE",
            "method": "ISOLATION_METHOD",
            "species": "SPECIES",
            "year": "YEAR",
            "disease": None,
            "vesicle": "VESICLE_TYPE",
            "characterization": None,
            "ev_metric": None,
            "source": "__FIXED__:Vesiclepedia",
            "source_type": "__FIXED__:cargo+experiment",
            "record_granularity": "__FIXED__:cargo_level"
        },
        "mapping_rationale": {
            "pmid": "Mapped to PUBMED_ID from experiment-level metadata.",
            "sample_name": "Mapped to SAMPLE_SOURCE as the most stable source/sample descriptor.",
            "working_id": "Coalesced cargo identifier (GENE_SYMBOL / MIRNA_ID / LIPID_ID) unified as WORKING_ID_RAW.",
            "molecule_type_raw": "Mapped to native CONTENT_TYPE across protein/mRNA, miRNA, and lipid cargo files.",
            "method": "Mapped to experiment-level ISOLATION_METHOD; cargo-level METHODS discarded to avoid semantic drift.",
            "species": "Mapped to experiment-level species (SPECIES_EXP); cargo-level SPECIES discarded for consistency.",
            "year": "Mapped to YEAR from experiment metadata.",
            "disease": "No semantically comparable disease field available; explicitly null-mapped.",
            "vesicle": "Mapped to VESICLE_TYPE as the native vesicle annotation field.",
            "characterization": "No semantically comparable field defined; explicitly null-mapped.",
            "ev_metric": "Not applicable; explicitly null-mapped.",
            "source": "Fixed source label.",
            "source_type": "Fixed source architecture label.",
            "record_granularity": "Fixed record-granularity label."
        }
    },

    "ExoCarta": {
        "source_type": "cargo+experiment",
        "record_granularity": "cargo_level",
        "experiment_file": EXOCARTA_EXPERIMENT,
        "join_key": "EXPERIMENT_ID",
        "required_experiment_headers": [
            "EXPERIMENT_ID", "PUBMED_ID", "SPECIES", "SAMPLE_SOURCE",
            "YEAR", "ISOLATION_METHOD"
        ],
        "canonical_map": {
            "pmid": "PUBMED_ID",
            "sample_name": "SAMPLE_SOURCE",
            "working_id": "WORKING_ID_RAW",
            "molecule_type_raw": "CONTENT_TYPE",
            "method": "ISOLATION_METHOD",
            "species": "SPECIES",
            "year": "YEAR",
            "disease": None,
            "vesicle": None,
            "characterization": None,
            "ev_metric": None,
            "source": "__FIXED__:ExoCarta",
            "source_type": "__FIXED__:cargo+experiment",
            "record_granularity": "__FIXED__:cargo_level"
        },
        "mapping_rationale": {
            "pmid": "Mapped to PUBMED_ID from experiment-level metadata.",
            "sample_name": "Mapped to SAMPLE_SOURCE as the most stable source/sample descriptor.",
            "working_id": "Coalesced cargo identifier (GENE_SYMBOL / MIRNA_ID / LIPID_ID) unified as WORKING_ID_RAW.",
            "molecule_type_raw": "Mapped to native CONTENT_TYPE across protein/mRNA, miRNA, and lipid cargo files.",
            "method": "Mapped to experiment-level ISOLATION_METHOD; cargo-level METHODS discarded to avoid semantic drift.",
            "species": "Mapped to experiment-level species (SPECIES_EXP); cargo-level SPECIES discarded for consistency.",
            "year": "Mapped to YEAR from experiment metadata.",
            "disease": "No semantically comparable disease field available; explicitly null-mapped.",
            "vesicle": "No semantically comparable vesicle field available; explicitly null-mapped.",
            "characterization": "No semantically comparable field defined; explicitly null-mapped.",
            "ev_metric": "Not applicable; explicitly null-mapped.",
            "source": "Fixed source label.",
            "source_type": "Fixed source architecture label.",
            "record_granularity": "Fixed record-granularity label."
        }
    },

    "EV-TRACK": {
        "source_type": "study-level",
        "record_granularity": "study_level",
        "experiment_file": EVTRACK_FILE,
        "required_headers": [
            "PMID", "SAMPLE_TYPE", "ISOLATION_PROTOCOLS", "SPECIES",
            "YEAR_OF_PUBLICATION", "SUBJECT_CONDITION",
            "VESILERELATED_TERM", "EV_METRIC_SCORE"
        ],
        "canonical_map": {
            "pmid": "PMID",
            "sample_name": "SAMPLE_TYPE",
            "working_id": None,
            "molecule_type_raw": None,
            "method": "ISOLATION_PROTOCOLS",
            "species": "SPECIES",
            "year": "YEAR_OF_PUBLICATION",
            "disease": "SUBJECT_CONDITION",
            "vesicle": "VESILERELATED_TERM",
            "characterization": None,
            "ev_metric": "EV_METRIC_SCORE",
            "source": "__FIXED__:EV-TRACK",
            "source_type": "__FIXED__:study-level",
            "record_granularity": "__FIXED__:study_level"
        },
        "mapping_rationale": {
            "pmid": "Mapped to PMID as the study-level publication identifier.",
            "sample_name": "Mapped to SAMPLE_TYPE as the available sample-related descriptor.",
            "working_id": "No cargo-level molecular identifier exists; explicitly null-mapped.",
            "molecule_type_raw": "No cargo-type field exists; explicitly null-mapped.",
            "method": "Mapped to ISOLATION_PROTOCOLS.",
            "species": "Mapped to SPECIES as the native organism field.",
            "year": "Mapped to YEAR_OF_PUBLICATION.",
            "disease": "Mapped to SUBJECT_CONDITION as the nearest condition field.",
            "vesicle": "Mapped to VESILERELATED_TERM after value-level inspection.",
            "characterization": "Not forced into current canonical schema; explicitly null-mapped.",
            "ev_metric": "Mapped to EV_METRIC_SCORE as the primary EV-METRIC field.",
            "source": "Fixed source label.",
            "source_type": "Fixed source architecture label.",
            "record_granularity": "Fixed record-granularity label."
        }
    }
}

CANONICAL_FIELDS = [
    "pmid", "sample_name", "working_id", "molecule_type_raw", "method",
    "species", "year", "disease", "vesicle", "characterization",
    "ev_metric", "source", "source_type", "record_granularity"
]

AUDIT_FIELDS = [
    "pmid", "sample_name", "working_id", "molecule_type_raw", "method",
    "species", "year", "disease", "vesicle", "characterization", "ev_metric"
]

# ===============================================================
# 1. UTILITY FUNCTIONS
# ===============================================================

def print_section(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)


def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "_", regex=False)
    )
    return df


def is_missing_like(val):
    if pd.isna(val):
        return True
    return str(val).strip().casefold() in MISSING_LIKE


def normalize_text_basic(val):
    if pd.isna(val):
        return "Unknown"
    v = str(val).strip()
    if v.casefold() in MISSING_LIKE:
        return "Unknown"
    return v


def normalize_pmid(val):
    if pd.isna(val):
        return "Unknown"
    v = str(val).strip()
    if v.casefold() in MISSING_LIKE:
        return "Unknown"
    if v.endswith(".0"):
        v = v[:-2]
    return v.strip()


def normalize_year(val):
    if pd.isna(val):
        return "Unknown"
    v = str(val).strip()
    if v.casefold() in MISSING_LIKE:
        return "Unknown"
    if v.endswith(".0"):
        v = v[:-2]
    return v


def normalize_species(val):
    if pd.isna(val):
        return "Unknown"

    species_dict = {
        "HUMAN": "Homo sapiens",
        "HOMO SAPIENS": "Homo sapiens",
        "MOUSE": "Mus musculus",
        "MUS MUSCULUS": "Mus musculus",
        "RAT": "Rattus norvegicus",
        "RATTUS NORVEGICUS": "Rattus norvegicus",
        "COW": "Bos taurus",
        "BOVINE": "Bos taurus",
        "BOS TAURUS": "Bos taurus",
        "PIG": "Sus scrofa",
        "PORCINE": "Sus scrofa",
        "SUS SCROFA": "Sus scrofa",
        "UNKNOWN": "Unknown"
    }

    v = str(val).strip()
    if v.casefold() in MISSING_LIKE:
        return "Unknown"
    return species_dict.get(v.upper(), v)


def normalize_molecule_type(val):
    """
    Deterministic lexical collapsing for analytical comparability.
    Not ontology-level synonym expansion.
    """
    if pd.isna(val):
        return "Unknown/Other"
    v = str(val).strip()
    if v.casefold() in MISSING_LIKE.union({"unknown", "not reported", "notfound"}):
        return "Unknown/Other"
    if v in {"0", "1"}:
        return "Unknown/Other"

    vl = v.lower()

    exact_map = {
        "protein": "Protein", "mrna": "mRNA", "mirna": "miRNA",
        "snorna": "snoRNA", "snrna": "snRNA", "scarna": "scaRNA",
        "lncrna": "lncRNA", "ncrna": "ncRNA", "vtrna": "vtRNA",
        "yrna": "yRNA", "lincrna": "lincRNA", "trna": "tRNA", "dna": "DNA",
    }
    if vl in exact_map:
        return exact_map[vl]

    if "lipid" in vl:
        return "Lipid"
    if "metabolite" in vl or "metabolomics" in vl:
        return "Metabolite"

    # Catch miRNA variants BEFORE the generic "rna" -> Other RNA rule,
    # so 'miRNAs', 'microRNA', 'micro RNA', 'miR' are not silently lost.
    if (vl in {"mirnas", "micrornas", "microrna", "mir", "mirs"}
            or vl.startswith("mirna") or vl.startswith("microrna")
            or "micro rna" in vl or "micro-rna" in vl):
        return "miRNA"

    if vl in {"mrnas"} or vl.startswith("mrna"):
        return "mRNA"

    if "rna" in vl:
        return "Other RNA"

    # Compound protein labels only (exact 'protein' already handled above).
    if vl.startswith("protein ") or vl.endswith(" protein") or " protein " in vl:
        return "Protein"

    return v


def group_molecule_type(val):
    v = str(val).strip()
    if v in {"Protein", "mRNA", "miRNA", "DNA", "Lipid", "Metabolite"}:
        return v
    if v in {
        "snoRNA", "snRNA", "scaRNA", "lncRNA", "ncRNA", "vtRNA",
        "yRNA", "lincRNA", "tRNA", "Other RNA"
    }:
        return "Other RNA"
    return "Unknown/Other"


def assert_required_headers(df, required_headers, source_name, table_name):
    missing = [h for h in required_headers if h not in df.columns]
    if missing:
        raise KeyError(
            f"[SCHEMA VALIDATION ERROR] Source='{source_name}', table='{table_name}' "
            f"is missing required headers: {missing}\nAvailable columns: {list(df.columns)}"
        )


def fixed_or_series(df, source_name, canonical_field, native_spec):
    if native_spec is None:
        return pd.Series([pd.NA] * len(df), index=df.index)

    if isinstance(native_spec, str) and native_spec.startswith("__FIXED__:"):
        fixed_value = native_spec.replace("__FIXED__:", "", 1)
        return pd.Series([fixed_value] * len(df), index=df.index)

    if native_spec not in df.columns:
        raise KeyError(
            f"[MAPPING ERROR] Source '{source_name}', canonical field '{canonical_field}' "
            f"expected exact native header '{native_spec}', but it was not found."
        )

    return df[native_spec]


def save_schema_inventory(df, source_name, table_name, out_rows):
    for idx, col in enumerate(df.columns):
        out_rows.append({
            "source": source_name,
            "table_name": table_name,
            "column_index": idx,
            "column_name": col
        })


def build_mapping_audit_rows(source_name, canonical_map, mapping_rationale, source_type):
    rows = []
    for field in CANONICAL_FIELDS:
        native_spec = canonical_map[field]
        if native_spec is None:
            selected_header = "None"
            mapping_type = "policy_null"
        elif isinstance(native_spec, str) and native_spec.startswith("__FIXED__:"):
            selected_header = native_spec.replace("__FIXED__:", "", 1)
            mapping_type = "fixed_value"
        else:
            selected_header = native_spec
            mapping_type = "exact_native_header"

        rows.append({
            "source": source_name,
            "source_type": source_type,
            "canonical_field": field,
            "selected_native_header_or_policy": selected_header,
            "mapping_type": mapping_type,
            "rationale": mapping_rationale.get(field, "")
        })
    return rows


def build_harmonized_frame(df, source_name, canonical_map):
    return pd.DataFrame({
        "pmid": fixed_or_series(df, source_name, "pmid", canonical_map["pmid"]),
        "sample_name": fixed_or_series(df, source_name, "sample_name", canonical_map["sample_name"]),
        "working_id": fixed_or_series(df, source_name, "working_id", canonical_map["working_id"]),
        "molecule_type_raw": fixed_or_series(df, source_name, "molecule_type_raw", canonical_map["molecule_type_raw"]),
        "method": fixed_or_series(df, source_name, "method", canonical_map["method"]),
        "species": fixed_or_series(df, source_name, "species", canonical_map["species"]),
        "year": fixed_or_series(df, source_name, "year", canonical_map["year"]),
        "disease": fixed_or_series(df, source_name, "disease", canonical_map["disease"]),
        "vesicle": fixed_or_series(df, source_name, "vesicle", canonical_map["vesicle"]),
        "characterization": fixed_or_series(df, source_name, "characterization", canonical_map["characterization"]),
        "ev_metric": fixed_or_series(df, source_name, "ev_metric", canonical_map["ev_metric"]),
        "source": fixed_or_series(df, source_name, "source", canonical_map["source"]),
        "source_type": fixed_or_series(df, source_name, "source_type", canonical_map["source_type"]),
        "record_granularity": fixed_or_series(df, source_name, "record_granularity", canonical_map["record_granularity"]),
    })


def add_source_trace_ids(df, source_name):
    df = df.copy()
    df["source_row_uid"] = [f"{source_name}::{i}" for i in range(len(df))]
    return df


def load_and_unify_cargo(source_name, base_dir):
    """
    Load all cargo files for a source, harmonize their identifier column
    into a single 'WORKING_ID_RAW' column, and concatenate.
    Rows whose CONTENT_TYPE is an unmappable placeholder (e.g. 'NOTFOUND')
    are dropped before harmonization and logged for the audit trail.
    Cargo-level SPECIES and METHODS columns are discarded so that all
    context comes from experiment-level metadata after the join.
    """
    frames = []
    dropped_log = []

    for spec in CARGO_FILES[source_name]:
        path = base_dir / spec["file"]
        cargo = clean_columns(pd.read_csv(path, sep="\t", dtype=str, low_memory=False))

        for required in ["EXPERIMENT_ID", "CONTENT_TYPE"]:
            if required not in cargo.columns:
                raise KeyError(
                    f"[{source_name}] cargo file '{spec['file']}' "
                    f"is missing required column '{required}'. "
                    f"Available: {list(cargo.columns)}"
                )

        id_col = spec["id_col"]
        if id_col not in cargo.columns:
            raise KeyError(
                f"[{source_name}] cargo file '{spec['file']}' "
                f"is missing declared id column '{id_col}'. "
                f"Available: {list(cargo.columns)}"
            )

        n_before = len(cargo)

        # Drop unmappable placeholder rows (e.g. Vesiclepedia miRNA 'NOTFOUND').
        ct_norm = cargo["CONTENT_TYPE"].fillna("").astype(str).str.strip().str.casefold()
        invalid_mask = ct_norm.isin(INVALID_CONTENT_TYPES)
        n_dropped = int(invalid_mask.sum())
        if n_dropped > 0:
            cargo = cargo.loc[~invalid_mask].copy()
            dropped_log.append({
                "source": source_name,
                "cargo_file": spec["file"],
                "rows_before": n_before,
                "rows_dropped_invalid_content_type": n_dropped,
                "rows_after": len(cargo),
            })
            print(f"  [{source_name}] {spec['file']}: dropped {n_dropped:,} "
                  f"invalid-CONTENT_TYPE rows (e.g. NOTFOUND).")

        # Unify identifier and keep only join-relevant cargo columns.
        cargo["WORKING_ID_RAW"] = cargo[id_col]
        keep = ["EXPERIMENT_ID", "CONTENT_TYPE", "WORKING_ID_RAW"]
        cargo = cargo[keep].copy()
        cargo["__cargo_file"] = spec["file"]
        frames.append(cargo)

        print(f"  [{source_name}] loaded {spec['file']}: {len(cargo):,} valid rows "
              f"(id_col={id_col}, content_types="
              f"{sorted(cargo['CONTENT_TYPE'].astype(str).str.lower().unique())})")

    unified = pd.concat(frames, ignore_index=True)
    print(f"  [{source_name}] total unified cargo rows: {len(unified):,}")
    return unified, pd.DataFrame(dropped_log)


def build_completeness_table(df, fields):
    rows = []
    n = len(df)
    for field in fields:
        s = df[field].astype(str).str.strip()
        unknown_mask = (s == "Unknown")
        rows.append({
            "field": field,
            "total_rows": n,
            "non_unknown_count": int((~unknown_mask).sum()),
            "unknown_count": int(unknown_mask.sum()),
            "non_unknown_pct": round(100 * (~unknown_mask).sum() / n, 3) if n else 0.0,
            "unknown_pct": round(100 * unknown_mask.sum() / n, 3) if n else 0.0,
            "unique_values": int(s.nunique(dropna=False))
        })
    return pd.DataFrame(rows)


def build_completeness_by_source(df, fields):
    rows = []
    for source_name, sub in df.groupby("source", dropna=False):
        n = len(sub)
        for field in fields:
            s = sub[field].astype(str).str.strip()
            unknown_mask = (s == "Unknown")
            rows.append({
                "source": source_name,
                "field": field,
                "total_rows": n,
                "non_unknown_count": int((~unknown_mask).sum()),
                "unknown_count": int(unknown_mask.sum()),
                "non_unknown_pct": round(100 * (~unknown_mask).sum() / n, 3) if n else 0.0,
                "unknown_pct": round(100 * unknown_mask.sum() / n, 3) if n else 0.0,
                "unique_values": int(s.nunique(dropna=False))
            })
    return pd.DataFrame(rows)


def build_structural_population_table(df, fields):
    rows = []
    n = len(df)
    for field in fields:
        s = df[field].astype(str).str.strip()
        unknown = (s == "Unknown")
        unknown_other = (s == "Unknown/Other")
        populated = ~(unknown | (s == ""))
        rows.append({
            "field": field,
            "total_rows": n,
            "structurally_populated_count": int(populated.sum()),
            "structurally_populated_pct": round(100 * populated.sum() / n, 3) if n else 0.0,
            "strict_unknown_count": int(unknown.sum()),
            "strict_unknown_pct": round(100 * unknown.sum() / n, 3) if n else 0.0,
            "unknown_or_other_count": int(unknown_other.sum()),
            "unknown_or_other_pct": round(100 * unknown_other.sum() / n, 3) if n else 0.0
        })
    return pd.DataFrame(rows)


def summarize_top_values(df, fields, source_col="source", top_n=15):
    rows = []
    for source_name in sorted(df[source_col].astype(str).unique()):
        sub = df[df[source_col] == source_name]
        for field in fields:
            vc = sub[field].astype(str).value_counts(dropna=False).head(top_n)
            for rank, (val, cnt) in enumerate(vc.items(), start=1):
                rows.append({
                    "source": source_name,
                    "field": field,
                    "rank": rank,
                    "value": val,
                    "count": int(cnt)
                })
    return pd.DataFrame(rows)


def build_molecule_mapping_audit(df, top_n=200):
    raw_counts = (
        df["molecule_type_raw"]
        .astype(str)
        .value_counts(dropna=False)
        .head(top_n)
        .rename_axis("molecule_type_raw")
        .reset_index(name="count")
    )
    rows = []
    for _, row in raw_counts.iterrows():
        raw_val = row["molecule_type_raw"]
        norm_val = normalize_molecule_type(raw_val)
        rows.append({
            "molecule_type_raw": raw_val,
            "count": int(row["count"]),
            "molecule_type_norm": norm_val,
            "molecule_type_group": group_molecule_type(norm_val)
        })
    return pd.DataFrame(rows)


def audit_join_coverage(merged_df, source_name, join_key, experiment_cols, cargo_rows_before):
    exp_key_missing = merged_df[join_key].isna().sum() if join_key in merged_df.columns else np.nan

    candidate_cols = [c for c in experiment_cols if c != join_key and c in merged_df.columns]
    representative = candidate_cols[0] if candidate_cols else None

    if representative is not None:
        unmatched = merged_df[representative].isna().sum()
    else:
        unmatched = np.nan

    matched = len(merged_df) - unmatched if pd.notna(unmatched) else np.nan
    match_rate = round(100 * matched / len(merged_df), 3) if len(merged_df) and pd.notna(matched) else np.nan

    return {
        "source": source_name,
        "cargo_rows_before_merge": int(cargo_rows_before),
        "merged_rows": int(len(merged_df)),
        "unmatched_rows_after_left_join": int(unmatched) if pd.notna(unmatched) else np.nan,
        "matched_rows_after_left_join": int(matched) if pd.notna(matched) else np.nan,
        "left_join_match_rate_pct": match_rate,
        "missing_join_key_after_merge": int(exp_key_missing) if pd.notna(exp_key_missing) else np.nan
    }


def build_dedup_sensitivity_table(df, key_dict):
    rows = []
    pre_n = len(df)
    for key_name, key_cols in key_dict.items():
        deduped = df.drop_duplicates(subset=key_cols).copy()
        post_n = len(deduped)
        rows.append({
            "dedup_key_label": key_name,
            "dedup_key_columns": "|".join(key_cols),
            "pre_dedup_rows": pre_n,
            "post_dedup_rows": post_n,
            "removed_rows": pre_n - post_n,
            "retention_pct": round(100 * post_n / pre_n, 3) if pre_n else 0.0
        })
    return pd.DataFrame(rows)


def build_collision_audit(df, dedup_key, top_n=250):
    grp = (
        df.groupby(dedup_key, dropna=False)
        .size()
        .reset_index(name="group_size")
        .sort_values(["group_size"], ascending=False)
    )
    collisions = grp[grp["group_size"] > 1].head(top_n).copy()
    if collisions.empty:
        return pd.DataFrame(columns=dedup_key + ["group_size", "species_variants", "disease_variants", "vesicle_variants", "year_variants"])

    merged = collisions.merge(df, on=dedup_key, how="left")
    rows = []
    for _, sub in merged.groupby(dedup_key, dropna=False):
        first = sub.iloc[0][dedup_key].to_dict()
        rows.append({
            **first,
            "group_size": int(len(sub)),
            "species_variants": int(sub["species"].astype(str).nunique(dropna=False)),
            "disease_variants": int(sub["disease"].astype(str).nunique(dropna=False)),
            "vesicle_variants": int(sub["vesicle"].astype(str).nunique(dropna=False)),
            "year_variants": int(sub["year"].astype(str).nunique(dropna=False))
        })
    return pd.DataFrame(rows).sort_values(
        ["group_size", "species_variants", "disease_variants", "vesicle_variants"],
        ascending=False
    )


def assert_no_literal_na_strings(df, df_name="dataframe"):
    forbidden = {"<NA>", "<na>"}
    observed = set(df.astype(str).stack().unique())
    bad = forbidden.intersection(observed)
    assert not bad, f"Literal NA strings detected in {df_name}: {bad}"


def assert_invariants(df_all, df_final, export_columns):
    assert len(df_final) <= len(df_all), "Deduplication increased row count unexpectedly."
    assert df_final["record_uid"].is_unique, "record_uid is not unique."
    assert df_final["source_row_uid"].is_unique, "source_row_uid is not unique in final table."
    assert set(export_columns).issubset(df_final.columns), "Missing export columns in final table."
    assert df_final["molecule_type_norm"].astype(str).eq("<NA>").sum() == 0, "molecule_type_norm still contains <NA>."
    assert_no_literal_na_strings(df_all, "df_all")
    assert_no_literal_na_strings(df_final, "df_final")


# ===============================================================
# 2. LOAD SOURCES + STRICT SCHEMA VALIDATION
# ===============================================================

schema_inventory_rows = []
mapping_audit_rows = []
join_audit_rows = []
dropped_cargo_frames = []

# -------------------------
# Vesiclepedia
# -------------------------
print_section("[Vesiclepedia] Loading and validating source tables")

vp_exp = clean_columns(pd.read_csv(BASE_DIR / MAPPING_SPEC["Vesiclepedia"]["experiment_file"], sep="\t", dtype=str, low_memory=False))
assert_required_headers(vp_exp, MAPPING_SPEC["Vesiclepedia"]["required_experiment_headers"], "Vesiclepedia", "experiment")
save_schema_inventory(vp_exp, "Vesiclepedia", "experiment", schema_inventory_rows)

vp_cargo, vp_dropped = load_and_unify_cargo("Vesiclepedia", BASE_DIR)
dropped_cargo_frames.append(vp_dropped)
save_schema_inventory(vp_cargo, "Vesiclepedia", "cargo_unified", schema_inventory_rows)

vp_join_key = MAPPING_SPEC["Vesiclepedia"]["join_key"]
vp_exp_before_dedup = len(vp_exp)
vp_exp = vp_exp.drop_duplicates(subset=[vp_join_key]).copy()
vp_exp_after_dedup = len(vp_exp)

vp_merged = pd.merge(vp_cargo, vp_exp, on=vp_join_key, how="left", suffixes=("_CARGO", "_EXP"))

join_audit_rows.append(
    audit_join_coverage(
        vp_merged, "Vesiclepedia", vp_join_key,
        MAPPING_SPEC["Vesiclepedia"]["required_experiment_headers"],
        cargo_rows_before=len(vp_cargo)
    )
)

mapping_audit_rows.extend(
    build_mapping_audit_rows(
        "Vesiclepedia",
        MAPPING_SPEC["Vesiclepedia"]["canonical_map"],
        MAPPING_SPEC["Vesiclepedia"]["mapping_rationale"],
        MAPPING_SPEC["Vesiclepedia"]["source_type"]
    )
)

df_vp = build_harmonized_frame(vp_merged, "Vesiclepedia", MAPPING_SPEC["Vesiclepedia"]["canonical_map"])
df_vp = add_source_trace_ids(df_vp, "Vesiclepedia")

print(f"[Vesiclepedia] Experiment rows before de-dup on join key: {vp_exp_before_dedup:,}")
print(f"[Vesiclepedia] Experiment rows after de-dup on join key:  {vp_exp_after_dedup:,}")
print(f"[Vesiclepedia] Harmonized rows: {len(df_vp):,}")

# -------------------------
# ExoCarta
# -------------------------
print_section("[ExoCarta] Loading and validating source tables")

exo_exp = clean_columns(pd.read_csv(BASE_DIR / MAPPING_SPEC["ExoCarta"]["experiment_file"], sep="\t", dtype=str, low_memory=False))
assert_required_headers(exo_exp, MAPPING_SPEC["ExoCarta"]["required_experiment_headers"], "ExoCarta", "experiment")
save_schema_inventory(exo_exp, "ExoCarta", "experiment", schema_inventory_rows)

exo_cargo, exo_dropped = load_and_unify_cargo("ExoCarta", BASE_DIR)
dropped_cargo_frames.append(exo_dropped)
save_schema_inventory(exo_cargo, "ExoCarta", "cargo_unified", schema_inventory_rows)

exo_join_key = MAPPING_SPEC["ExoCarta"]["join_key"]
exo_exp_before_dedup = len(exo_exp)
exo_exp = exo_exp.drop_duplicates(subset=[exo_join_key]).copy()
exo_exp_after_dedup = len(exo_exp)

exo_merged = pd.merge(exo_cargo, exo_exp, on=exo_join_key, how="left", suffixes=("_CARGO", "_EXP"))

join_audit_rows.append(
    audit_join_coverage(
        exo_merged, "ExoCarta", exo_join_key,
        MAPPING_SPEC["ExoCarta"]["required_experiment_headers"],
        cargo_rows_before=len(exo_cargo)
    )
)

mapping_audit_rows.extend(
    build_mapping_audit_rows(
        "ExoCarta",
        MAPPING_SPEC["ExoCarta"]["canonical_map"],
        MAPPING_SPEC["ExoCarta"]["mapping_rationale"],
        MAPPING_SPEC["ExoCarta"]["source_type"]
    )
)

df_exo = build_harmonized_frame(exo_merged, "ExoCarta", MAPPING_SPEC["ExoCarta"]["canonical_map"])
df_exo = add_source_trace_ids(df_exo, "ExoCarta")

print(f"[ExoCarta] Experiment rows before de-dup on join key: {exo_exp_before_dedup:,}")
print(f"[ExoCarta] Experiment rows after de-dup on join key:  {exo_exp_after_dedup:,}")
print(f"[ExoCarta] Harmonized rows: {len(df_exo):,}")

# -------------------------
# EV-TRACK
# -------------------------
print_section("[EV-TRACK] Loading and validating source table")

ev_raw = clean_columns(pd.read_excel(BASE_DIR / MAPPING_SPEC["EV-TRACK"]["experiment_file"], dtype=str))
assert_required_headers(ev_raw, MAPPING_SPEC["EV-TRACK"]["required_headers"], "EV-TRACK", "table")
save_schema_inventory(ev_raw, "EV-TRACK", "table", schema_inventory_rows)

mapping_audit_rows.extend(
    build_mapping_audit_rows(
        "EV-TRACK",
        MAPPING_SPEC["EV-TRACK"]["canonical_map"],
        MAPPING_SPEC["EV-TRACK"]["mapping_rationale"],
        MAPPING_SPEC["EV-TRACK"]["source_type"]
    )
)

df_ev = build_harmonized_frame(ev_raw, "EV-TRACK", MAPPING_SPEC["EV-TRACK"]["canonical_map"])
df_ev = add_source_trace_ids(df_ev, "EV-TRACK")

print(f"[EV-TRACK] Harmonized rows: {len(df_ev):,}")

# ===============================================================
# 3. CONCATENATION + FIELD CLEANING
# ===============================================================

print_section("Concatenating and cleaning harmonized tables")

df_all = pd.concat([df_vp, df_exo, df_ev], ignore_index=True)

for col in CANONICAL_FIELDS:
    df_all[col] = df_all[col].apply(normalize_text_basic)

df_all["pmid"] = df_all["pmid"].apply(normalize_pmid)
df_all["year"] = df_all["year"].apply(normalize_year)
df_all["species"] = df_all["species"].apply(normalize_species)

df_all["molecule_type_raw"] = df_all["molecule_type_raw"].apply(
    lambda x: "Unknown" if pd.isna(x) else str(x).strip()
)
df_all["molecule_type_norm"] = df_all["molecule_type_raw"].apply(normalize_molecule_type)
df_all["molecule_type_group"] = df_all["molecule_type_norm"].apply(group_molecule_type)
df_all["molecule_type"] = df_all["molecule_type_norm"]

df_all["pre_dedup_uid"] = [f"MASTER_PRE::{i}" for i in range(len(df_all))]

assert_no_literal_na_strings(df_all, "df_all_after_cleaning")

print(f"Concatenated rows before deduplication: {len(df_all):,}")

# ===============================================================
# 4. PROVENANCE-PRESERVING EXACT DEDUPLICATION
# ===============================================================

print_section("Applying provenance-preserving exact deduplication (key_B_plus_species)")

pre_counts = df_all["source"].value_counts(dropna=False).rename_axis("source").reset_index(name="pre_dedup_rows")

df_final = df_all.drop_duplicates(subset=PRIMARY_DEDUP_KEY).copy()
df_final = df_final.reset_index(drop=True)
df_final["record_uid"] = [f"EVisionary_keyB::{i}" for i in range(len(df_final))]

post_counts = df_final["source"].value_counts(dropna=False).rename_axis("source").reset_index(name="post_dedup_rows")
dedup_summary = pre_counts.merge(post_counts, on="source", how="outer").fillna(0)
dedup_summary["removed_rows"] = dedup_summary["pre_dedup_rows"] - dedup_summary["post_dedup_rows"]
dedup_summary["retention_pct"] = np.where(
    dedup_summary["pre_dedup_rows"] > 0,
    np.round(100 * dedup_summary["post_dedup_rows"] / dedup_summary["pre_dedup_rows"], 3),
    0.0
)

overall_pre = len(df_all)
overall_post = len(df_final)
overall_removed = overall_pre - overall_post
overall_retention = round(100 * overall_post / overall_pre, 3) if overall_pre else 0.0
overall_reduction = round(100 * overall_removed / overall_pre, 3) if overall_pre else 0.0

print(f"Final rows after deduplication: {len(df_final):,}")
print(f"Overall retention: {overall_retention:.3f}%")
print(f"Overall reduction: {overall_reduction:.3f}%")

# ===============================================================
# 5. SENSITIVITY + COLLISION AUDITS
# ===============================================================

print_section("Running dedup sensitivity and collision audits")

dedup_sensitivity = build_dedup_sensitivity_table(df_all, DEDUP_KEY_SENSITIVITY)
collision_audit = build_collision_audit(df_all, PRIMARY_DEDUP_KEY, top_n=250)

# ===============================================================
# 6. AUDIT TABLES
# ===============================================================

print_section("Building audit and supplementary-ready tables")

table_schema_inventory = pd.DataFrame(schema_inventory_rows)
table_mapping_audit = pd.DataFrame(mapping_audit_rows)
table_join_audit = pd.DataFrame(join_audit_rows)

# Dropped invalid cargo (e.g. NOTFOUND) audit
if dropped_cargo_frames:
    table_dropped_cargo = pd.concat(dropped_cargo_frames, ignore_index=True)
else:
    table_dropped_cargo = pd.DataFrame(
        columns=["source", "cargo_file", "rows_before",
                 "rows_dropped_invalid_content_type", "rows_after"]
    )

table_completeness_overall = build_completeness_table(df_final, AUDIT_FIELDS)
table_completeness_by_source = build_completeness_by_source(df_final, AUDIT_FIELDS)
table_structural_population = build_structural_population_table(df_final, AUDIT_FIELDS)

table_top_value_audit = summarize_top_values(
    df_final,
    fields=[
        "molecule_type_raw", "molecule_type_norm", "molecule_type_group",
        "species", "method", "vesicle", "disease", "sample_name", "ev_metric"
    ],
    top_n=15
)

table_molecule_mapping_audit = build_molecule_mapping_audit(df_all, top_n=200)
table_dedup_summary = dedup_summary.copy()

table_source_record_counts = (
    df_final["source"].value_counts(dropna=False)
    .rename_axis("source").reset_index(name="final_rows")
)

table_granularity_counts = (
    df_final["record_granularity"].value_counts(dropna=False)
    .rename_axis("record_granularity").reset_index(name="final_rows")
)

table_molecule_group_counts = (
    df_final["molecule_type_group"].value_counts(dropna=False)
    .rename_axis("molecule_type_group").reset_index(name="final_rows")
)

table_overall_dedup_metrics = pd.DataFrame([{
    "primary_dedup_key_label": "key_B_plus_species",
    "primary_dedup_key_columns": "|".join(PRIMARY_DEDUP_KEY),
    "pre_dedup_rows": overall_pre,
    "post_dedup_rows": overall_post,
    "removed_rows": overall_removed,
    "retention_pct": overall_retention,
    "reduction_pct": overall_reduction
}])

# Export
table_schema_inventory.to_csv(AUDIT_DIR / "Audit_source_schema_inventory.csv", index=False)
# Export audit tables
table_schema_inventory.to_csv(AUDIT_DIR / "Audit_source_schema_inventory.csv", index=False)
table_mapping_audit.to_csv(AUDIT_DIR / "Audit_canonical_mapping_summary.csv", index=False)
table_join_audit.to_csv(AUDIT_DIR / "Audit_left_join_coverage.csv", index=False)
table_dropped_cargo.to_csv(AUDIT_DIR / "Audit_dropped_invalid_cargo.csv", index=False)

table_completeness_overall.to_csv(AUDIT_DIR / "Audit_field_completeness_overall.csv", index=False)
table_completeness_by_source.to_csv(AUDIT_DIR / "Audit_field_completeness_by_source.csv", index=False)
table_structural_population.to_csv(AUDIT_DIR / "Audit_structural_population_vs_unknown.csv", index=False)

table_top_value_audit.to_csv(AUDIT_DIR / "Audit_top_value_semantic_audit.csv", index=False)
table_molecule_mapping_audit.to_csv(AUDIT_DIR / "Audit_molecule_type_raw_to_normalized.csv", index=False)

table_dedup_summary.to_csv(AUDIT_DIR / "Audit_primary_dedup_summary.csv", index=False)
table_overall_dedup_metrics.to_csv(AUDIT_DIR / "Audit_primary_dedup_overall_metrics.csv", index=False)
dedup_sensitivity.to_csv(AUDIT_DIR / "Audit_dedup_key_sensitivity.csv", index=False)
collision_audit.to_csv(AUDIT_DIR / "Audit_dedup_collision_review.csv", index=False)

table_source_record_counts.to_csv(AUDIT_DIR / "Audit_final_record_counts_by_source.csv", index=False)
table_granularity_counts.to_csv(AUDIT_DIR / "Audit_final_record_counts_by_granularity.csv", index=False)
table_molecule_group_counts.to_csv(AUDIT_DIR / "Audit_final_record_counts_by_molecule_group.csv", index=False)

# ===============================================================
# 7. FINAL CONSOLE AUDIT
# ===============================================================

print_section("Final console audit")

print("\n=== Dropped invalid cargo (e.g. NOTFOUND) ===")
if len(table_dropped_cargo):
    print(table_dropped_cargo.to_string(index=False))
else:
    print("None dropped.")

print("\n=== Final row counts by source ===")
print(table_source_record_counts.to_string(index=False))

print("\n=== Final row counts by granularity ===")
print(table_granularity_counts.to_string(index=False))

print("\n=== Final row counts by molecule group ===")
print(table_molecule_group_counts.to_string(index=False))

print("\n=== Join coverage audit ===")
print(table_join_audit.to_string(index=False))

print("\n=== Primary dedup summary ===")
print(table_dedup_summary.to_string(index=False))

print("\n=== Overall dedup metrics ===")
print(table_overall_dedup_metrics.to_string(index=False))

print("\n=== Dedup sensitivity ===")
print(dedup_sensitivity.to_string(index=False))

print("\n=== Overall completeness ===")
print(table_completeness_overall.to_string(index=False))

print("\n=== Structural population audit ===")
print(table_structural_population.to_string(index=False))

print("\n=== molecule_type_norm (top 25) ===")
print(df_final["molecule_type_norm"].value_counts(dropna=False).head(25).to_string())

print("\n=== molecule_type_group (full) ===")
print(df_final["molecule_type_group"].value_counts(dropna=False).to_string())

print("\n=== species (top 20) ===")
print(df_final["species"].value_counts(dropna=False).head(20).to_string())

print("\n=== molecule_type_norm by source ===")
print(
    df_final.groupby("source")["molecule_type_norm"]
    .value_counts(dropna=False)
    .groupby(level=0, group_keys=False)
    .head(8)
    .to_string()
)

# ===============================================================
# 8. EXPORT FINAL MASTER TABLE
# ===============================================================

print_section("Exporting final harmonized master table")

EXPORT_COLUMNS = [
    "record_uid",
    "source_row_uid",
    "pre_dedup_uid",
    "pmid",
    "sample_name",
    "working_id",
    "molecule_type_raw",
    "molecule_type_norm",
    "molecule_type_group",
    "molecule_type",
    "method",
    "species",
    "year",
    "disease",
    "vesicle",
    "characterization",
    "ev_metric",
    "source",
    "source_type",
    "record_granularity"
]

df_final = df_final[EXPORT_COLUMNS].copy()

assert_invariants(df_all, df_final, EXPORT_COLUMNS)

df_final.to_parquet(OUTPUT_PARQUET, index=False)

print("\n" + "=" * 88)
print(" PIPELINE COMPLETED SUCCESSFULLY")
print(f" Final master-table records: {len(df_final):,}")
print(f" Exported Parquet: {OUTPUT_PARQUET}")
print(f" Audit tables exported to: {AUDIT_DIR}")
print(f" Final columns: {list(df_final.columns)}")
print("=" * 88)

 Initiating EVisionary Database Harmonization (key_B, multi-cargo)

[Vesiclepedia] Loading and validating source tables
  [Vesiclepedia] loaded Vesiclepedia_protein_mRNA_details.txt: 537,765 valid rows (id_col=GENE_SYMBOL, content_types=['mrna', 'protein', 'protein ', 'snrna'])
  [Vesiclepedia] VESICLEPEDIA_MIRNA_DETAILS_5.1.txt: dropped 12,337 invalid-CONTENT_TYPE rows (e.g. NOTFOUND).
  [Vesiclepedia] loaded VESICLEPEDIA_MIRNA_DETAILS_5.1.txt: 22,858 valid rows (id_col=MIRNA_ID, content_types=['mirna'])
  [Vesiclepedia] loaded VESICLEPEDIA_LIPID_DETAILS_5.1.txt: 4,031 valid rows (id_col=LIPID_ID, content_types=['lipid'])
  [Vesiclepedia] total unified cargo rows: 564,654
[Vesiclepedia] Experiment rows before de-dup on join key: 3,481
[Vesiclepedia] Experiment rows after de-dup on join key:  3,481
[Vesiclepedia] Harmonized rows: 564,654

[ExoCarta] Loading and validating source tables
  [ExoCarta] ExoCarta_protein_mRNA_details_6.txt: dropped 2 invalid-CONTENT_TYPE rows (e.g. NOTFOUND)